# 00 - Corrección de Ruido Eléctrico en Grabaciones

Detecta y elimina pulsos de ruido eléctrico (broadband, alta energía).

**Pipeline:**
1. Análisis visual de un archivo de ejemplo
2. Detección automática por RMS + spectral flatness
3. Limpieza (silenciado de frames contaminados)
4. Procesado batch de todos los audios

In [ ]:
import numpy as np
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Audio, display
from scipy.ndimage import binary_dilation
from tqdm.notebook import tqdm

# ─────────────────────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────────────────────
AUDIO_DIR   = Path('../audios')          # directorio con audios originales
CLEAN_DIR   = Path('../audios_clean')    # directorio de salida
CLEAN_DIR.mkdir(exist_ok=True)

# Detección de ruido
FRAME_LENGTH  = 2048    # ~46ms a 44100 Hz
HOP_LENGTH    = 512     # ~12ms
RMS_K         = 3.0     # umbral RMS: media + K * std
FLATNESS_K    = 2.0     # umbral spectral flatness: media + K * std
REQUIRE_BOTH  = True    # True: noisy si RMS *y* flatness altos

# Post-procesado
MARGIN_FRAMES = 3       # frames de margen alrededor del ruido detectado
FADE_SAMPLES  = 256     # muestras de fade-in/out en bordes del silencio

print('Configuración cargada.')

## 1. Diagnóstico Visual

Carga un audio de ejemplo y visualiza forma de onda, espectrograma y energía RMS.

In [ ]:
example_files = sorted(AUDIO_DIR.glob('*.wav'))
if not example_files:
    raise FileNotFoundError(f'No hay archivos WAV en {AUDIO_DIR}')

EXAMPLE_FILE = example_files[0]  # ← cambia por un archivo con ruido conocido
print(f'Archivo de ejemplo: {EXAMPLE_FILE.name}')

y, sr = librosa.load(EXAMPLE_FILE, sr=None, mono=True)

fig, axes = plt.subplots(3, 1, figsize=(14, 9))

librosa.display.waveshow(y, sr=sr, ax=axes[0], color='steelblue', alpha=0.8)
axes[0].set_title('Forma de Onda')

D = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH)), ref=np.max)
img = librosa.display.specshow(D, sr=sr, hop_length=HOP_LENGTH, x_axis='time', y_axis='hz', ax=axes[1], cmap='inferno')
axes[1].set_title('Espectrograma')
fig.colorbar(img, ax=axes[1], format='%+2.0f dB')

rms = librosa.feature.rms(y=y, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]
times = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=HOP_LENGTH)
rms_thresh = np.mean(rms) + RMS_K * np.std(rms)
axes[2].plot(times, rms, color='darkorange', linewidth=0.8)
axes[2].axhline(rms_thresh, color='red', linestyle='--', label=f'Umbral RMS (mu+{RMS_K}*sigma) = {rms_thresh:.4f}')
axes[2].set_title('Energia RMS por Frame')
axes[2].set_xlabel('Tiempo (s)')
axes[2].legend()

plt.tight_layout()
plt.show()
print(f'Duracion: {len(y)/sr:.2f}s | SR: {sr} Hz | Frames sobre umbral RMS: {np.sum(rms > rms_thresh)}')

## 2. Funciones de Detección y Limpieza

In [ ]:
def detect_noise_mask(y, sr, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH,
                      rms_k=RMS_K, flatness_k=FLATNESS_K, require_both=REQUIRE_BOTH,
                      margin_frames=MARGIN_FRAMES):
    """
    Devuelve mascara booleana a nivel de muestra: True = muestra contaminada.
    Criterio: RMS alto + spectral flatness alta (broadband noise).
    """
    n_frames = 1 + (len(y) - frame_length) // hop_length
    rms      = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0][:n_frames]
    flatness = librosa.feature.spectral_flatness(y=y, n_fft=frame_length, hop_length=hop_length)[0][:n_frames]

    rms_thresh      = np.mean(rms) + rms_k * np.std(rms)
    flatness_thresh = np.mean(flatness) + flatness_k * np.std(flatness)

    noisy_rms  = rms > rms_thresh
    noisy_flat = flatness > flatness_thresh
    noisy_frames = (noisy_rms & noisy_flat) if require_both else (noisy_rms | noisy_flat)

    if margin_frames > 0:
        noisy_frames = binary_dilation(noisy_frames, iterations=margin_frames)

    # frames -> muestras
    sample_mask = np.zeros(len(y), dtype=bool)
    for i, is_noisy in enumerate(noisy_frames):
        if is_noisy:
            s = i * hop_length
            e = min(s + frame_length, len(y))
            sample_mask[s:e] = True

    return sample_mask, rms, flatness, rms_thresh, flatness_thresh


def clean_audio(y, sr, **kwargs):
    """
    Silencia segmentos con ruido electrico. Aplica fade en bordes.
    Devuelve (audio_limpio, n_frames_ruidosos, fraccion_silenciada).
    """
    sample_mask, rms, flatness, rms_t, flat_t = detect_noise_mask(y, sr, **kwargs)
    y_clean = y.copy()

    changes = np.diff(sample_mask.astype(int))
    starts  = np.where(changes == 1)[0] + 1
    ends    = np.where(changes == -1)[0] + 1
    if sample_mask[0]:  starts = np.concatenate([[0], starts])
    if sample_mask[-1]: ends   = np.concatenate([ends, [len(y)]])

    for s, e in zip(starts, ends):
        fs = FADE_SAMPLES
        if s > 0:
            s0 = max(0, s - fs)
            y_clean[s0:s] *= np.linspace(1.0, 0.0, s - s0)
        if e < len(y):
            e1 = min(len(y), e + fs)
            y_clean[e:e1] *= np.linspace(0.0, 1.0, e1 - e)
        y_clean[s:e] = 0.0

    n_noisy = int(np.sum(rms > rms_t))
    frac    = float(np.mean(sample_mask))
    return y_clean, n_noisy, frac


print('Funciones definidas.')

## 3. Validación en el Archivo de Ejemplo

In [ ]:
y, sr = librosa.load(EXAMPLE_FILE, sr=None, mono=True)
y_clean, n_noisy, frac = clean_audio(y, sr)

print(f'Archivo: {EXAMPLE_FILE.name}')
print(f'Frames RMS ruidosos: {n_noisy}')
print(f'Porcion silenciada: {frac*100:.2f}%')

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

librosa.display.waveshow(y, sr=sr, ax=axes[0,0], color='steelblue', alpha=0.7)
axes[0,0].set_title('Waveform Original')
librosa.display.waveshow(y_clean, sr=sr, ax=axes[0,1], color='green', alpha=0.7)
axes[0,1].set_title('Waveform Limpio')

D1 = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH)), ref=np.max)
D2 = librosa.amplitude_to_db(np.abs(librosa.stft(y_clean, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH)), ref=np.max)
librosa.display.specshow(D1, sr=sr, hop_length=HOP_LENGTH, x_axis='time', y_axis='hz', ax=axes[1,0], cmap='inferno')
axes[1,0].set_title('Espectrograma Original')
librosa.display.specshow(D2, sr=sr, hop_length=HOP_LENGTH, x_axis='time', y_axis='hz', ax=axes[1,1], cmap='inferno')
axes[1,1].set_title('Espectrograma Limpio')

plt.tight_layout()
plt.show()

print('\n--- Audio ORIGINAL ---')
display(Audio(data=y, rate=sr))
print('--- Audio LIMPIO ---')
display(Audio(data=y_clean, rate=sr))

## 4. Diagnóstico Detallado de Detección

Visualiza exactamente qué frames se detectan como ruidosos.
Ajusta `RMS_K` y `FLATNESS_K` en la celda de configuración si el resultado no es correcto:
- **K bajo** → más agresivo (puede silenciar audio útil)
- **K alto** → solo detecta picos extremos
- **REQUIRE_BOTH = False** → silencia si cualquiera de los dos criterios se cumple

In [ ]:
y, sr = librosa.load(EXAMPLE_FILE, sr=None, mono=True)
sample_mask, rms, flatness, rms_t, flat_t = detect_noise_mask(y, sr)

noisy_times = librosa.frames_to_time(np.where(rms > rms_t)[0], sr=sr, hop_length=HOP_LENGTH)
all_times   = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=HOP_LENGTH)

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

D = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH)), ref=np.max)
librosa.display.specshow(D, sr=sr, hop_length=HOP_LENGTH, x_axis='time', y_axis='hz', ax=axes[0], cmap='inferno')
for t in noisy_times:
    axes[0].axvline(t, color='cyan', alpha=0.5, linewidth=1)
axes[0].set_title('Espectrograma (cyan = frames ruidosos detectados)')

axes[1].plot(all_times, rms, color='darkorange', linewidth=0.7)
axes[1].axhline(rms_t, color='red', linestyle='--', label=f'Umbral RMS={rms_t:.4f}')
axes[1].set_title('Energia RMS'); axes[1].legend()

axes[2].plot(all_times, flatness, color='purple', linewidth=0.7)
axes[2].axhline(flat_t, color='red', linestyle='--', label=f'Umbral Flatness={flat_t:.4f}')
axes[2].set_title('Spectral Flatness (broadband noise -> valor alto)'); axes[2].legend()
axes[2].set_xlabel('Tiempo (s)')

plt.tight_layout()
plt.show()

## 5. Procesado Batch

In [ ]:
import pandas as pd

audio_files = sorted(AUDIO_DIR.glob('*.wav'))
print(f'Archivos a procesar: {len(audio_files)}')

stats = []
skipped = []

for f in tqdm(audio_files, desc='Limpiando audios'):
    out_path = CLEAN_DIR / f.name
    if out_path.exists() and out_path.stat().st_mtime > f.stat().st_mtime:
        skipped.append(f.name)
        continue
    try:
        y, sr = librosa.load(f, sr=None, mono=False)
        if y.ndim == 1:
            y_clean, n_noisy, frac = clean_audio(y, sr)
        else:
            channels_clean, n_noisy = [], 0
            for ch in range(y.shape[0]):
                yc, nn, frac = clean_audio(y[ch], sr)
                channels_clean.append(yc)
                n_noisy += nn
            y_clean = np.stack(channels_clean)
        sf.write(out_path, y_clean.T if y_clean.ndim > 1 else y_clean, sr)
        stats.append({'file': f.name, 'n_noisy_frames': n_noisy, 'frac_silenced': round(frac, 4)})
    except Exception as e:
        print(f'[ERROR] {f.name}: {e}')

print(f'Procesados: {len(stats)} | Ya existian: {len(skipped)}')

df_stats = pd.DataFrame(stats)
if not df_stats.empty:
    noisy_files = df_stats[df_stats['frac_silenced'] > 0].sort_values('frac_silenced', ascending=False)
    print(f'Archivos con ruido detectado: {len(noisy_files)}')
    display(noisy_files.head(20))
    print(f'Media fraccion silenciada: {df_stats["frac_silenced"].mean()*100:.2f}%')
    df_stats.to_csv('../data/processed/audio_cleaning_stats.csv', index=False)
    print('Stats guardadas en data/processed/audio_cleaning_stats.csv')

## 6. Verificación Final

In [ ]:
if not df_stats.empty and df_stats['frac_silenced'].max() > 0:
    worst = df_stats.sort_values('frac_silenced', ascending=False).iloc[0]
    print(f'Archivo con mayor ruido: {worst["file"]} ({worst["frac_silenced"]*100:.1f}% silenciado)')

    y_orig, sr = librosa.load(AUDIO_DIR / worst['file'], sr=None, mono=True)
    y_cln,  _  = librosa.load(CLEAN_DIR / worst['file'], sr=None, mono=True)

    fig, axes = plt.subplots(2, 2, figsize=(16, 7))
    librosa.display.waveshow(y_orig, sr=sr, ax=axes[0,0], color='steelblue')
    axes[0,0].set_title('Waveform Original')
    librosa.display.waveshow(y_cln, sr=sr, ax=axes[0,1], color='green')
    axes[0,1].set_title('Waveform Limpio')
    D1 = librosa.amplitude_to_db(np.abs(librosa.stft(y_orig)), ref=np.max)
    D2 = librosa.amplitude_to_db(np.abs(librosa.stft(y_cln)), ref=np.max)
    librosa.display.specshow(D1, sr=sr, x_axis='time', y_axis='hz', ax=axes[1,0], cmap='inferno')
    axes[1,0].set_title('Espectrograma Original')
    librosa.display.specshow(D2, sr=sr, x_axis='time', y_axis='hz', ax=axes[1,1], cmap='inferno')
    axes[1,1].set_title('Espectrograma Limpio')
    plt.tight_layout()
    plt.show()

    print('\nOriginal:')
    display(Audio(data=y_orig, rate=sr))
    print('Limpio:')
    display(Audio(data=y_cln, rate=sr))
else:
    print('No se detecto ruido significativo. Baja RMS_K o FLATNESS_K en la celda de configuracion.')